# 02 - Condition だけを名寄せして curated h5ad + sidecar CSV を保存

このノートブックは **手入力の作業スペース** です。`pandas / numpy / scanpy / anndata / pathlib`
だけで動くシンプルな構成にしています。

## このノートブックの方針

- この 02 では **Condition の名寄せだけ** を行う。cell type / genotype / treatment /
  disease_status などの整理はまだ行わない。
- curated h5ad の `obs` は merge / QC に最低限必要な列だけにする。
- 元の `adata.obs` の全カラムは `original_obs_metadata_by_cell.csv`（sidecar CSV）に保存する。
- 細胞アノテーションなどは、後から `cell_uid` をキーに sidecar CSV から戻す。
- QC や可視化で必要になった列だけ sidecar CSV から追加する。
- `cell_uid` は `GSE番号 + "__" + 元の adata.obs_names` で作る。
- `cell_uid` は merge 後も sidecar CSV と結合するための **最重要キー** である。
- `Condition_original` は元データでの表記、`Condition` は解析用に揃えた表記である。

## curated h5ad の obs に残す列

```
cell_uid:
  全データセットを通して各細胞を一意に識別するID。
  元の adata.obs_names に GSE番号をprefixとして付けて作る。
  merge後や sidecar CSV との結合に使う最重要キー。
  例: GSE206330__AAACCTGAGATCGATA-1

source_accession:
  その細胞が由来するGSE番号。
  例: GSE206330, GSE173524

dataset_id:
  そのh5adファイルまたはデータセットを識別するID。
  基本的には入力h5adのファイル名stemを使う。
  例: GSE206330_sc_cortex_sod1_facs_glia_soupx

cell_id_original:
  元のh5adを読んだ時点の adata.obs_names。
  元データ内での細胞ID。
  GSE間では重複しうるので、mergeやsidecar結合には cell_uid を使う。

Condition_original:
  Condition名寄せに使った元の値。
  例: sample_label の "WT_1"、Condition列の "SOD1"、gsm_id の "GSM510xxxx" など。
  どの列を使うかは CURATION_RULES[acc]["condition_source"] で指定する。

Condition:
  解析で使うために名寄せした条件名。
  例: WT, Ctrl, SOD1, SMA, SOD1_RIPK1i, unknown。
  CURATION_RULES[acc]["condition_map"] で Condition_original から変換する。

sample_id:
  サンプル・個体・ライブラリなどを識別するID。
  既存obsに sample_id があれば使う。
  なければ "unknown"。

sample_label:
  元データや01で付いたサンプル名・ラベル。
  既存obsに sample_label があれば使う。
  なければ "unknown"。
```

以下の列は curated h5ad には追加せず、すべて sidecar CSV に逃がす:
`cell_type, cell_type_original, author_cell_type, author_condition, genotype,
treatment, disease_status, data_status`。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

# v2 ルートを探す（config/dataset_manifest.yaml がある場所）
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

INTERIM_DIR = ROOT / "data" / "interim_h5ad"
CURATED_DIR = ROOT / "data" / "curated_h5ad"
CURATED_DIR.mkdir(parents=True, exist_ok=True)

print("project root:", ROOT)
print("interim dir :", INTERIM_DIR)
print("curated dir :", CURATED_DIR)

project root: /home/suzuki/Learn/SMA/v2
interim dir : /home/suzuki/Learn/SMA/v2/data/interim_h5ad
curated dir : /home/suzuki/Learn/SMA/v2/data/curated_h5ad


## interim h5ad の一覧

In [2]:
interim_files = sorted(p for p in INTERIM_DIR.glob("*.h5ad"))
print(f"{len(interim_files)} interim h5ad files found:\n")
for p in interim_files:
    print(p.name)


def acc_of(path):
    """ファイル名の先頭 GSE 番号を accession として返す（例: GSE287569_xxx.h5ad -> GSE287569）。"""
    return path.stem.split("_")[0]


def load_interim_by_acc(acc):
    """accession（例: "GSE287569"）を prefix に持つ interim h5ad を読み込む。"""
    matches = sorted(p for p in INTERIM_DIR.glob(f"{acc}*.h5ad"))
    if not matches:
        raise FileNotFoundError(f"no interim h5ad starting with {acc!r} in {INTERIM_DIR}")
    if len(matches) > 1:
        print(f"[warn] multiple files match {acc!r}, using first: {[p.name for p in matches]}")
    return sc.read_h5ad(matches[0])

11 interim h5ad files found:

GSE167198_sc_spinalcord_sod1_dropseq.h5ad
GSE167327_sc_spinalcord_optn_cd45_enriched_indrop.h5ad
GSE167331_sc_spinalcord_optn_facs_microglia_smartseq2_tpm.h5ad
GSE173524_sn_lumbar_spinalcord_sod1.h5ad
GSE178693_sc_brainstem_trigeminal_sod1_dropseq.h5ad
GSE206330_sc_cortex_sod1_facs_glia_soupx.h5ad
GSE208629_sc_spinalcord_sma.h5ad
GSE219201_sn_lumbar_spinalcord_sod1_phd1.h5ad
GSE242939_sc_spinalcord_sod1_pf04457845.h5ad
GSE287569_sc_spinalcord_sod1_ripk1i.h5ad
GSE295514_sc_brain_rnls8_tdp43_rds.h5ad


## obs 確認用の関数

In [3]:
def inspect_obs(adata, max_values=50):
    print(adata)
    print("\nobs columns:")
    for i, col in enumerate(adata.obs.columns):
        print(f"{i}: {col}")

    for col in adata.obs.columns:
        s = adata.obs[col]
        n = s.dropna().astype(str).nunique()
        vals = s.dropna().astype(str).unique().tolist()[:max_values]
        print(f"\n--- {col} ---")
        print("n_unique:", n)
        print(vals)

## GSE を 1 つ指定して obs を確認する（作業セル）

`acc` を書き換えて、各 GSE の obs を順番に確認し、下の `CURATION_RULES` を埋めてください。

In [25]:
acc = "GSE295514"
adata = load_interim_by_acc(acc)
inspect_obs(adata)

AnnData object with n_obs × n_vars = 123401 × 24134
    obs: 'cell_id_original', 'sample_id', 'sample_label', 'gsm_id', 'source_file', 'source_accession', 'dataset_id', 'meta_orig.ident', 'meta_nCount_RNA', 'meta_nFeature_RNA'
    var: 'gene_symbol', 'gene_symbol_upper', 'gene_id', 'ensembl_id', 'feature_type'

obs columns:
0: cell_id_original
1: sample_id
2: sample_label
3: gsm_id
4: source_file
5: source_accession
6: dataset_id
7: meta_orig.ident
8: meta_nCount_RNA
9: meta_nFeature_RNA

--- cell_id_original ---
n_unique: 123401
['GEX_S1_ALS-FC1_AAACCCAAGAAACCCG-1', 'GEX_S1_ALS-FC1_AAACCCAAGCGTGCTC-1', 'GEX_S1_ALS-FC1_AAACCCAAGGAGTCTG-1', 'GEX_S1_ALS-FC1_AAACCCAAGGTAAGGA-1', 'GEX_S1_ALS-FC1_AAACCCAAGTGGCCTC-1', 'GEX_S1_ALS-FC1_AAACCCACACAACGTT-1', 'GEX_S1_ALS-FC1_AAACCCACACCAGTAT-1', 'GEX_S1_ALS-FC1_AAACCCACACTTGGGC-1', 'GEX_S1_ALS-FC1_AAACCCACAGAAGTTA-1', 'GEX_S1_ALS-FC1_AAACCCACAGCGAACA-1', 'GEX_S1_ALS-FC1_AAACCCACATGGTGGA-1', 'GEX_S1_ALS-FC1_AAACCCAGTACCTTCC-1', 'GEX_S1_ALS-FC1_AAA

In [26]:
adata.obs["meta_orig.ident"].unique()

['GEX_S1_ALS-FC1', 'GEX_S2_ALS-MC1', 'GEX_S3_ALS-F1', 'GEX_S4_ALS-M1', 'GEX_S5_ALS-FC2', 'GEX_S6_ALS-MC2', 'GEX_S7_ALS-F2', 'GEX_S8_ALS-M2']
Categories (8, object): ['GEX_S1_ALS-FC1', 'GEX_S2_ALS-MC1', 'GEX_S3_ALS-F1', 'GEX_S4_ALS-M1', 'GEX_S5_ALS-FC2', 'GEX_S6_ALS-MC2', 'GEX_S7_ALS-F2', 'GEX_S8_ALS-M2']

## CURATION_RULES（Condition 名寄せの手入力スペース）

`CURATION_RULES` は、各GSEでどのobs列をCondition名寄せに使うか、そして元の値をどの標準Condition名に変換するかを書く作業スペースである。

例:
```
  condition_source = "sample_label"
  condition_map = {"WT_1": "WT", "WT_2": "WT", "SOD1_1": "SOD1"}
```

この場合:
```
  Condition_original は WT_1, WT_2, SOD1_1
  Condition は WT, WT, SOD1
```

- `condition_source` が `None` のときは `Condition_original = "unknown"`, `Condition = "unknown"`。
- `condition_map` で map されなかった値は `"unknown"` にし、warning を表示する。
- 今回は完成させなくてよい（空辞書のままで構わない）。

In [27]:
CURATION_RULES = {
    "GSE287569": {
        "condition_source": "sample_label",
        "condition_map": {
            'SOD1-s1':"WT", 
            'SOD1-s2':"WT", 
            'SOD1-s3':"SOD1", 
            'SOD1-s4':"SOD1", 
            'SOD1-s5':"SOD1", 
            'SOD1-s6':"SOD1", 
            'SOD1-s7':"WT", 
            'SOD1-s8':"WT", 
            'SOD1-s9':"SOD1", 
            'SOD1-s10':"SOD1", 
            'SOD1-s11':"SOD1", 
            'SOD1-s12':"SOD1"
        },
    },

    "GSE173524": {
        "condition_source": "meta_Genotype",
        "condition_map": {
            "WT": "WT",
            "SOD1G93A": "SOD1",
        },
    }, #finish

    "GSE167198": {
        "condition_source": "gsm_id",
        "condition_map": {
            'GSM5098907':"WT", 
            'GSM5098908':"WT", 
            'GSM5098909':"SOD1", 
            'GSM5098910':"SOD1", 
            'GSM5098911':"SOD1", 
            'GSM5098912':"SOD1"
        },
    },#fin

    "GSE167327": {
        "condition_source": "sample_label", #違うけどとりあえず
        "condition_map": {
            "GSE167327_CD45_enriched":"SOD1"
            },
    },

    "GSE167331": {
        "condition_source": "sample_label",
        "condition_map": {
            "GSE167331":"SOD1"
        },
    },

    "GSE219201": {
        "condition_source": "sample_label",
        "condition_map": {
            'WT_1':"WT",
            'WT_2':"WT", 
            'WT_3':"WT", 
            'Phd1-KO_1':"SOD1", 
            'Phd1-KO_2':"SOD1", 
            'Phd1-KO_3':"SOD1"
        },
    },

    "GSE242939": {
        "condition_source": "sample_label",
        "condition_map": {
            'CON'"": 'WT', 
            'PF':"SOD1"
        },
    },#fin

    "GSE206330": {
        "condition_source": "Condition",
        "condition_map": {
            'CTRL':"WT",
            "SOD1": "SOD1",
        },
    },

    "GSE178693": {
        "condition_source": "sample_label",
        "condition_map": {
            'WT1': 'WT',
            'WT2': 'WT',
            'Mutant1': 'SOD1',
            'Mutant2': 'SOD1'
        },
    },

    "GSE295514": {
        "condition_source": "meta_orig.ident",
        "condition_map": {
            'GEX_S1_ALS-FC1':"WT", 
            'GEX_S2_ALS-MC1':"WT", 
            'GEX_S3_ALS-F1':"SOD1", 
            'GEX_S4_ALS-M1':"SOD1", 
            'GEX_S5_ALS-FC2':"WT", 
            'GEX_S6_ALS-MC2':"WT", 
            'GEX_S7_ALS-F2':"SOD1",
            'GEX_S8_ALS-M2':"SOD1"
        },
    },

    "GSE208629": {
        "condition_source": "sample_label",
        "condition_map": {
            'SMA':"SMA", 
            'Con':"WT"
        },
    },
}

print(f"{len(CURATION_RULES)} GSE entries in CURATION_RULES")

11 GSE entries in CURATION_RULES


## apply_curation（Condition のみ処理）

In [28]:
def apply_curation(adata, acc, input_file, rules):
    adata = adata.copy()

    if not adata.obs_names.is_unique:
        raise ValueError(f"{acc}: adata.obs_names is not unique. Please fix this before curation.")

    dataset_id = input_file.stem
    cell_id_original = adata.obs_names.astype(str)
    cell_uid = acc + "__" + cell_id_original

    # Condition_original and Condition
    condition_source = rules.get(acc, {}).get("condition_source")
    condition_map = rules.get(acc, {}).get("condition_map", {})

    if condition_source is None:
        condition_original = pd.Series("unknown", index=adata.obs_names)
        condition_standardized = pd.Series("unknown", index=adata.obs_names)
    else:
        if condition_source not in adata.obs.columns:
            print(f"WARNING: {acc}: condition_source '{condition_source}' not found in obs")
            condition_original = pd.Series("unknown", index=adata.obs_names)
            condition_standardized = pd.Series("unknown", index=adata.obs_names)
        else:
            condition_original = adata.obs[condition_source].astype(str)
            condition_standardized = condition_original.map(condition_map).fillna("unknown")

            unmapped = sorted(condition_original[condition_standardized == "unknown"].dropna().unique().tolist())
            if unmapped:
                print(f"WARNING: {acc}: unmapped Condition_original values:", unmapped)

    sample_id = adata.obs["sample_id"].astype(str) if "sample_id" in adata.obs.columns else pd.Series("unknown", index=adata.obs_names)
    sample_label = adata.obs["sample_label"].astype(str) if "sample_label" in adata.obs.columns else pd.Series("unknown", index=adata.obs_names)

    minimal_obs = pd.DataFrame(index=adata.obs_names)
    minimal_obs["cell_uid"] = cell_uid
    minimal_obs["source_accession"] = acc
    minimal_obs["dataset_id"] = dataset_id
    minimal_obs["cell_id_original"] = cell_id_original
    minimal_obs["Condition_original"] = condition_original.values
    minimal_obs["Condition"] = condition_standardized.values
    minimal_obs["sample_id"] = sample_id.values
    minimal_obs["sample_label"] = sample_label.values

    adata.obs = minimal_obs
    adata.obs_names = adata.obs["cell_uid"].astype(str)

    return adata

## sidecar（元 obs 全カラム）を作る関数

In [31]:
def build_sidecar(adata, acc, dataset_id):
    """元の adata.obs 全カラムを保存しつつ、結合キーを先頭に付けた sidecar を返す。

    adata は curation 前（元の obs_names を持つ状態）で渡すこと。

    注意:
    元obsに cell_uid/source_accession/dataset_id などの管理列と同名の列がある場合、
    元obs側の列は orig_ prefix を付けて残す。
    """
    if not adata.obs_names.is_unique:
        raise ValueError(f"{acc}: adata.obs_names is not unique. Please fix this before curation.")

    original_obs = adata.obs.copy()

    cell_id_original = pd.Index(adata.obs_names).astype(str)
    cell_uid = pd.Index(acc + "__" + cell_id_original.astype(str)).astype(str)

    reserved_cols = {
        "cell_uid",
        "source_accession",
        "dataset_id",
        "cell_id_original",
        "obs_name_original",
        "obs_name_curated",
    }

    rename_map = {
        col: f"orig_{col}"
        for col in original_obs.columns
        if col in reserved_cols
    }

    if rename_map:
        print(f"WARNING: {acc}: renamed original obs columns in sidecar: {rename_map}")
        original_obs = original_obs.rename(columns=rename_map)

    key_df = pd.DataFrame(
        {
            "cell_uid": cell_uid.to_numpy(),
            "source_accession": acc,
            "dataset_id": dataset_id,
            "cell_id_original": cell_id_original.to_numpy(),
            "obs_name_original": cell_id_original.to_numpy(),
            "obs_name_curated": cell_uid.to_numpy(),
        },
        index=adata.obs_names,
    )

    sidecar = pd.concat([key_df, original_obs], axis=1)

    return sidecar.reset_index(drop=True)

## 全 GSE に curation を適用して保存（curated h5ad + sidecar CSV）

In [32]:
def _vals_str(values, limit=30):
    vals = sorted(pd.Series(values).astype(str).unique().tolist())
    shown = vals[:limit]
    s = ", ".join(shown)
    if len(vals) > limit:
        s += f", ... (+{len(vals) - limit} more)"
    return s


summary_rows = []
sidecar_rows = []

for p in interim_files:
    acc = acc_of(p)
    print(f"\n=== {acc}  ({p.name}) ===")

    adata = sc.read_h5ad(p)
    dataset_id = p.stem

    # sidecar は curation 前（元 obs）から作る
    sidecar_rows.append(build_sidecar(adata, acc, dataset_id))

    # Condition のみ名寄せ
    cur = apply_curation(adata, acc, p, CURATION_RULES)

    out_path = CURATED_DIR / p.name
    cur.write_h5ad(out_path)
    print(f"saved -> {out_path.name}  ({cur.n_obs} cells x {cur.n_vars} genes)")

    rule = CURATION_RULES.get(acc, {})
    obs = cur.obs
    summary_rows.append({
        "source_accession": acc,
        "input_file": p.name,
        "output_file": out_path.name,
        "n_cells": cur.n_obs,
        "n_genes": cur.n_vars,
        "condition_source": rule.get("condition_source"),
        "n_condition_original": obs["Condition_original"].nunique(),
        "condition_original_values": _vals_str(obs["Condition_original"]),
        "n_condition": obs["Condition"].nunique(),
        "condition_values": _vals_str(obs["Condition"]),
        "n_unknown_condition": int((obs["Condition"].astype(str) == "unknown").sum()),
    })

# sidecar CSV（元 obs 全カラム）を結合して保存
original_obs_metadata = pd.concat(sidecar_rows, axis=0, ignore_index=True)
sidecar_path = CURATED_DIR / "original_obs_metadata_by_cell.csv"
original_obs_metadata.to_csv(sidecar_path, index=False)
print(f"\nsidecar saved -> {sidecar_path.name}  "
      f"({original_obs_metadata.shape[0]} cells x {original_obs_metadata.shape[1]} cols)")

summary = pd.DataFrame(summary_rows)
print(f"done: {len(summary)} datasets curated")


=== GSE167198  (GSE167198_sc_spinalcord_sod1_dropseq.h5ad) ===


saved -> GSE167198_sc_spinalcord_sod1_dropseq.h5ad  (21919 cells x 19777 genes)

=== GSE167327  (GSE167327_sc_spinalcord_optn_cd45_enriched_indrop.h5ad) ===
saved -> GSE167327_sc_spinalcord_optn_cd45_enriched_indrop.h5ad  (2125 cells x 33666 genes)

=== GSE167331  (GSE167331_sc_spinalcord_optn_facs_microglia_smartseq2_tpm.h5ad) ===
saved -> GSE167331_sc_spinalcord_optn_facs_microglia_smartseq2_tpm.h5ad  (326 cells x 23420 genes)

=== GSE173524  (GSE173524_sn_lumbar_spinalcord_sod1.h5ad) ===
saved -> GSE173524_sn_lumbar_spinalcord_sod1.h5ad  (14662 cells x 31040 genes)

=== GSE178693  (GSE178693_sc_brainstem_trigeminal_sod1_dropseq.h5ad) ===
saved -> GSE178693_sc_brainstem_trigeminal_sod1_dropseq.h5ad  (15472 cells x 23903 genes)

=== GSE206330  (GSE206330_sc_cortex_sod1_facs_glia_soupx.h5ad) ===
saved -> GSE206330_sc_cortex_sod1_facs_glia_soupx.h5ad  (21580 cells x 46983 genes)

=== GSE208629  (GSE208629_sc_spinalcord_sma.h5ad) ===
saved -> GSE208629_sc_spinalcord_sma.h5ad  (52340 cell

## curation summary の表示と保存

In [33]:
summary_path = CURATED_DIR / "curation_summary.csv"
summary.to_csv(summary_path, index=False)
print("saved:", summary_path)
summary

saved: /home/suzuki/Learn/SMA/v2/data/curated_h5ad/curation_summary.csv


,source_accession,input_file,output_file,n_cells,n_genes,condition_source,n_condition_original,condition_original_values,n_condition,condition_values,n_unknown_condition
0,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq.h5ad,GSE167198_sc_spinalcord_sod1_dropseq.h5ad,21919,19777,gsm_id,6,"GSM5098907, GSM5098908, GSM5098909, GSM5098910...",2,"SOD1, WT",0
1,GSE167327,GSE167327_sc_spinalcord_optn_cd45_enriched_ind...,GSE167327_sc_spinalcord_optn_cd45_enriched_ind...,2125,33666,sample_label,1,GSE167327_CD45_enriched,1,SOD1,0
2,GSE167331,GSE167331_sc_spinalcord_optn_facs_microglia_sm...,GSE167331_sc_spinalcord_optn_facs_microglia_sm...,326,23420,sample_label,1,GSE167331,1,SOD1,0
3,GSE173524,GSE173524_sn_lumbar_spinalcord_sod1.h5ad,GSE173524_sn_lumbar_spinalcord_sod1.h5ad,14662,31040,meta_Genotype,2,"SOD1G93A, WT",2,"SOD1, WT",0
4,GSE178693,GSE178693_sc_brainstem_trigeminal_sod1_dropseq...,GSE178693_sc_brainstem_trigeminal_sod1_dropseq...,15472,23903,sample_label,4,"Mutant1, Mutant2, WT1, WT2",2,"SOD1, WT",0
5,GSE206330,GSE206330_sc_cortex_sod1_facs_glia_soupx.h5ad,GSE206330_sc_cortex_sod1_facs_glia_soupx.h5ad,21580,46983,Condition,2,"CTRL, SOD1",2,"SOD1, WT",0
6,GSE208629,GSE208629_sc_spinalcord_sma.h5ad,GSE208629_sc_spinalcord_sma.h5ad,52340,32285,sample_label,2,"Con, SMA",2,"SMA, WT",0
7,GSE219201,GSE219201_sn_lumbar_spinalcord_sod1_phd1.h5ad,GSE219201_sn_lumbar_spinalcord_sod1_phd1.h5ad,42758,32289,sample_label,6,"Phd1-KO_1, Phd1-KO_2, Phd1-KO_3, WT_1, WT_2, WT_3",2,"SOD1, WT",0
8,GSE242939,GSE242939_sc_spinalcord_sod1_pf04457845.h5ad,GSE242939_sc_spinalcord_sod1_pf04457845.h5ad,1877260,32285,sample_label,2,"CON, PF",2,"SOD1, WT",0
9,GSE287569,GSE287569_sc_spinalcord_sod1_ripk1i.h5ad,GSE287569_sc_spinalcord_sod1_ripk1i.h5ad,15854,32285,sample_label,12,"SOD1_s1, SOD1_s10, SOD1_s11, SOD1_s12, SOD1_s2...",1,unknown,15854


## sidecar CSV から元 obs 列を戻す関数

In [34]:
def attach_original_obs(adata, sidecar_csv, key="cell_uid", columns=None, prefix="orig_"):
    """sidecar CSV から元 obs 列を、key（既定 cell_uid）で join して戻す。

    - columns=None なら sidecar の全列を戻す。columns=[...] なら指定列だけ。
    - 戻す列には prefix を付け、curated 側の列と衝突させない。
    - sidecar に無い列が指定されたら warning を出し、存在する列だけ戻す。
    - join できなかった細胞数を表示する。
    - 元の adata は破壊せず copy を返す。
    """
    adata = adata.copy()
    sidecar = pd.read_csv(sidecar_csv, dtype=str)

    if key not in sidecar.columns:
        raise ValueError(f"key '{key}' not found in sidecar CSV")
    if not sidecar[key].is_unique:
        raise ValueError(f"{key} is not unique in sidecar CSV")
    if key not in adata.obs.columns:
        raise ValueError(f"key '{key}' not found in adata.obs")

    # 戻す列を決める（key 自体は除く）
    available = [c for c in sidecar.columns if c != key]
    if columns is None:
        use_cols = available
    else:
        missing = [c for c in columns if c not in available]
        if missing:
            print("WARNING: columns not in sidecar CSV (skipped):", missing)
        use_cols = [c for c in columns if c in available]

    sidecar = sidecar.set_index(key)
    keys = adata.obs[key].astype(str)

    matched = keys.isin(sidecar.index)
    n_unmatched = int((~matched).sum())
    print(f"attach_original_obs: {int(matched.sum())} matched, {n_unmatched} unmatched of {adata.n_obs} cells")

    for c in use_cols:
        adata.obs[f"{prefix}{c}"] = sidecar[c].reindex(keys.values).values

    return adata

## 再付与テスト（sidecar から元 obs を戻せることの確認 / 保存はしない）

In [35]:
sidecar_csv = CURATED_DIR / "original_obs_metadata_by_cell.csv"
test_file = sorted(CURATED_DIR.glob("*.h5ad"))[0]

test_adata = sc.read_h5ad(test_file)

# 全列を戻す
test_adata2 = attach_original_obs(
    test_adata,
    sidecar_csv=sidecar_csv,
    key="cell_uid",
    columns=None,
    prefix="orig_",
)

assert test_adata2.n_obs == test_adata.n_obs
assert "cell_uid" in test_adata2.obs.columns

orig_cols = [c for c in test_adata2.obs.columns if c.startswith("orig_")]
print("Number of restored original obs columns:", len(orig_cols))
print(orig_cols[:30])
display(test_adata2.obs[["cell_uid"] + orig_cols[:10]].head())

attach_original_obs: 21919 matched, 0 unmatched of 21919 cells


/tmp/ipykernel_1442393/2744016078.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adata.obs[f"{prefix}{c}"] = sidecar[c].reindex(keys.values).values
/tmp/ipykernel_1442393/2744016078.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adata.obs[f"{prefix}{c}"] = sidecar[c].reindex(keys.values).values
/tmp/ipykernel_1442393/2744016078.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u

Number of restored original obs columns: 179
['orig_source_accession', 'orig_dataset_id', 'orig_cell_id_original', 'orig_obs_name_original', 'orig_obs_name_curated', 'orig_orig_cell_id_original', 'orig_sample_id', 'orig_sample_label', 'orig_gsm_id', 'orig_source_file', 'orig_orig_source_accession', 'orig_orig_dataset_id', 'orig_cell_id', 'orig_parent_gse', 'orig_species', 'orig_disease_area', 'orig_tissue', 'orig_region', 'orig_assay', 'orig_technology', 'orig_enrichment', 'orig_disease_model', 'orig_data_status', 'orig_processing_status', 'orig_condition_original', 'orig_disease_status', 'orig_genotype', 'orig_treatment', 'orig_age', 'orig_age_month']


,cell_uid,orig_source_accession,orig_dataset_id,orig_cell_id_original,orig_obs_name_original,orig_obs_name_curated,orig_orig_cell_id_original,orig_sample_id,orig_sample_label,orig_gsm_id,orig_source_file
cell_uid,,,,,,,,,,,
GSE167198__GSE167198_Sample2-matrix-txt-gz_GCACATCCGCCG,GSE167198__GSE167198_Sample2-matrix-txt-gz_GCA...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,GSE167198_Sample2-matrix-txt-gz_GCACATCCGCCG,GSE167198_Sample2-matrix-txt-gz_GCACATCCGCCG,GSE167198__GSE167198_Sample2-matrix-txt-gz_GCA...,GCACATCCGCCG,Sample2-matrix-txt-gz,Sample2_matrix.txt.gz,GSM5098907,GSM5098907_Sample2_matrix.txt.gz
GSE167198__GSE167198_Sample2-matrix-txt-gz_CCCGAAAAGTAT,GSE167198__GSE167198_Sample2-matrix-txt-gz_CCC...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,GSE167198_Sample2-matrix-txt-gz_CCCGAAAAGTAT,GSE167198_Sample2-matrix-txt-gz_CCCGAAAAGTAT,GSE167198__GSE167198_Sample2-matrix-txt-gz_CCC...,CCCGAAAAGTAT,Sample2-matrix-txt-gz,Sample2_matrix.txt.gz,GSM5098907,GSM5098907_Sample2_matrix.txt.gz
GSE167198__GSE167198_Sample2-matrix-txt-gz_TCATATTAATAA,GSE167198__GSE167198_Sample2-matrix-txt-gz_TCA...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,GSE167198_Sample2-matrix-txt-gz_TCATATTAATAA,GSE167198_Sample2-matrix-txt-gz_TCATATTAATAA,GSE167198__GSE167198_Sample2-matrix-txt-gz_TCA...,TCATATTAATAA,Sample2-matrix-txt-gz,Sample2_matrix.txt.gz,GSM5098907,GSM5098907_Sample2_matrix.txt.gz
GSE167198__GSE167198_Sample2-matrix-txt-gz_TCGGACAAAGGC,GSE167198__GSE167198_Sample2-matrix-txt-gz_TCG...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,GSE167198_Sample2-matrix-txt-gz_TCGGACAAAGGC,GSE167198_Sample2-matrix-txt-gz_TCGGACAAAGGC,GSE167198__GSE167198_Sample2-matrix-txt-gz_TCG...,TCGGACAAAGGC,Sample2-matrix-txt-gz,Sample2_matrix.txt.gz,GSM5098907,GSM5098907_Sample2_matrix.txt.gz
GSE167198__GSE167198_Sample2-matrix-txt-gz_CCATAGGTTACN,GSE167198__GSE167198_Sample2-matrix-txt-gz_CCA...,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,GSE167198_Sample2-matrix-txt-gz_CCATAGGTTACN,GSE167198_Sample2-matrix-txt-gz_CCATAGGTTACN,GSE167198__GSE167198_Sample2-matrix-txt-gz_CCA...,CCATAGGTTACN,Sample2-matrix-txt-gz,Sample2_matrix.txt.gz,GSM5098907,GSM5098907_Sample2_matrix.txt.gz


In [36]:
# 細胞アノテーション候補だけ戻す例（存在しない列は warning のみ）
test_adata3 = attach_original_obs(
    test_adata,
    sidecar_csv=sidecar_csv,
    key="cell_uid",
    columns=[
        "cell_type",
        "meta_cell.type.refined",
        "meta_Genotype",
        "Condition",
        "sex",
        "subcluster",
        "nCount_RNA",
        "nFeature_RNA",
        "percent.mt",
    ],
    prefix="orig_",
)

display(test_adata3.obs.filter(like="orig_").head())

attach_original_obs: 21919 matched, 0 unmatched of 21919 cells


,orig_cell_type,orig_meta_cell.type.refined,orig_meta_Genotype,orig_Condition,orig_sex,orig_nCount_RNA,orig_nFeature_RNA,orig_percent.mt
cell_uid,,,,,,,,
GSE167198__GSE167198_Sample2-matrix-txt-gz_GCACATCCGCCG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSE167198__GSE167198_Sample2-matrix-txt-gz_CCCGAAAAGTAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSE167198__GSE167198_Sample2-matrix-txt-gz_TCATATTAATAA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSE167198__GSE167198_Sample2-matrix-txt-gz_TCGGACAAAGGC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSE167198__GSE167198_Sample2-matrix-txt-gz_CCATAGGTTACN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## (optional) merge preview

curated h5ad の `obs_names` はすでに `cell_uid` なので、追加の prefix は付けない（二重 prefix 防止）。**保存はしない**。

In [37]:
# Optional: simple merge preview
curated_files = sorted(CURATED_DIR.glob("*.h5ad"))
adatas = [sc.read_h5ad(p) for p in curated_files]

merged = ad.concat(
    adatas,
    join="outer",
    label="source_file",
    keys=[p.stem for p in curated_files],
    index_unique=None,
)

print(merged)
merged.obs[["source_accession", "dataset_id", "Condition"]].value_counts().head(50)

AnnData object with n_obs × n_vars = 2187697 × 56420
    obs: 'cell_uid', 'source_accession', 'dataset_id', 'cell_id_original', 'Condition_original', 'Condition', 'sample_id', 'sample_label', 'source_file'


source_accession  dataset_id                                                 Condition
GSE242939         GSE242939_sc_spinalcord_sod1_pf04457845                    SOD1         1191772
                                                                             WT            685488
GSE295514         GSE295514_sc_brain_rnls8_tdp43_rds                         SOD1           64436
                                                                             WT             58965
GSE208629         GSE208629_sc_spinalcord_sma                                SMA            26921
                                                                             WT             25419
GSE219201         GSE219201_sn_lumbar_spinalcord_sod1_phd1                   WT             22589
                                                                             SOD1           20169
GSE287569         GSE287569_sc_spinalcord_sod1_ripk1i                        unknown        15854
GSE167198         GSE167198_sc_